In [1]:
import pandas as pd
import numpy as np
import requests
import io
import pickle
import os
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)

In [2]:
url = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+ps&format=csv'
print('Fetching exoplanet data from NASA archive...')
response = requests.get(url, timeout=180)
df_raw = pd.read_csv(io.StringIO(response.text), comment='#', low_memory=False)
print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head(3)

Fetching exoplanet data from NASA archive...


Raw dataset shape: (39537, 355)


,pl_name,pl_letter,hostname,hd_name,hip_name,tic_id,gaia_dr2_id,gaia_dr3_id,default_flag,pl_refname,...,sy_jmagerr1,sy_jmagerr2,sy_jmagstr,sy_hmag,sy_hmagerr1,sy_hmagerr2,sy_hmagstr,sy_kmag,sy_kmagerr1,sy_kmagerr2
0,HD 156279 b,b,HD 156279,HD 156279,HIP 84171,TIC 232606278,Gaia DR2 1631084478574318976,Gaia DR3 1631084478574318976,0,<a refstr=DIAZ_ET_AL__2012 href=https://ui.ads...,...,0.018,-0.018,6.677&plusmn;0.018,6.346,0.026,-0.026,6.346&plusmn;0.026,6.269,0.023,-0.023
1,GJ 832 b,b,GJ 832,HD 204961,HIP 106440,TIC 139754153,Gaia DR2 6562924609150908416,Gaia DR3 6562924609150908416,0,<a refstr=BONFILS_ET_AL__2013 href=https://ui....,...,0.032,-0.032,5.349&plusmn;0.032,4.766,0.256,-0.256,4.766&plusmn;0.256,4.501,0.018,-0.018
2,GJ 1214 b,b,GJ 1214,NaN,NaN,TIC 467929202,Gaia DR2 4393265392167891712,Gaia DR3 4393265392168829056,0,<a refstr=CARTER_ET_AL__2011 href=https://ui.a...,...,0.024,-0.024,9.750&plusmn;0.024,9.094,0.024,-0.024,9.094&plusmn;0.024,8.782,0.020,-0.020


In [3]:
BASE_FEATURES = [
    'pl_orbper', 'pl_rade', 'pl_masse', 'pl_orbsmax',
    'pl_orbeccen', 'st_teff', 'st_rad', 'st_mass', 'sy_dist'
]
available_base = [c for c in BASE_FEATURES if c in df_raw.columns]
print('Available base features:', available_base)

df = df_raw[available_base].copy()
df = df.dropna(thresh=int(len(available_base) * 0.55))
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print(f'After filtering: {df.shape}')

Available base features: ['pl_orbper', 'pl_rade', 'pl_masse', 'pl_orbsmax', 'pl_orbeccen', 'st_teff', 'st_rad', 'st_mass', 'sy_dist']
After filtering: (32903, 9)


In [4]:
def add_derived_features(df):
    out = df.copy()
    st_rad  = out['st_rad'].fillna(1.0).clip(lower=0.01)
    st_teff = out['st_teff'].fillna(5778.0).clip(lower=500)
    a_au    = out['pl_orbsmax'].clip(lower=1e-4)

    st_rad_au  = st_rad * 0.00465
    luminosity = ((st_teff / 5778.0) ** 4) * (st_rad ** 2)

    out['t_eq']        = st_teff * np.sqrt(st_rad_au / (2.0 * a_au)) * (0.7 ** 0.25)
    out['stellar_flux']= luminosity / (a_au ** 2)
    hz_center          = np.sqrt(luminosity).clip(lower=1e-6)
    out['hz_ratio']    = a_au / hz_center
    return out

df = add_derived_features(df)

DERIVED_FEATURES = ['t_eq', 'stellar_flux', 'hz_ratio']
all_features = available_base + DERIVED_FEATURES
print('All features for model:', all_features)

All features for model: ['pl_orbper', 'pl_rade', 'pl_masse', 'pl_orbsmax', 'pl_orbeccen', 'st_teff', 'st_rad', 'st_mass', 'sy_dist', 't_eq', 'stellar_flux', 'hz_ratio']


In [5]:
def compute_habitability(df):
    t_eq_ok   = (df['t_eq'] >= 175) & (df['t_eq'] <= 340)
    radius_ok = (df['pl_rade'] >= 0.5) & (df['pl_rade'] <= 2.5)
    flux_ok   = (df['stellar_flux'] >= 0.2) & (df['stellar_flux'] <= 2.5)
    ecc_ok    = df['pl_orbeccen'].fillna(0.0) <= 0.4
    star_ok   = (df['st_teff'] >= 3500) & (df['st_teff'] <= 7500)

    if 'pl_masse' in df.columns:
        mass_proxy = df['pl_masse'].fillna(df['pl_rade'] ** 2.5)
    else:
        mass_proxy = df['pl_rade'] ** 2.5
    mass_ok = mass_proxy <= 13.0

    return (t_eq_ok & radius_ok & flux_ok & mass_ok & ecc_ok & star_ok).astype(int)

df['habitable'] = compute_habitability(df)
counts = df['habitable'].value_counts().sort_index()
print('Class distribution:')
print(counts)
print(f'Habitable fraction: {df["habitable"].mean():.4f}')

if counts.get(1, 0) < 10:
    raise ValueError(f'Too few habitable samples ({counts.get(1,0)}). Check dataset.')

Class distribution:
habitable
0    32694
1      209
Name: count, dtype: int64
Habitable fraction: 0.0064


In [6]:
n_cols = 4
plot_features = available_base
n_rows = (len(plot_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes_flat = axes.flatten()
for i, col in enumerate(plot_features):
    data = df[col].dropna()
    axes_flat[i].hist(data, bins=40, color='#2980b9', edgecolor='white', alpha=0.85)
    axes_flat[i].axvline(data.median(), color='#e74c3c', linestyle='--', linewidth=1.5,
                         label=f'Median: {data.median():.2g}')
    axes_flat[i].set_title(col, fontsize=10, fontweight='bold')
    axes_flat[i].legend(fontsize=7)
    axes_flat[i].set_xlabel('')
for j in range(len(plot_features), len(axes_flat)):
    axes_flat[j].set_visible(False)
plt.suptitle('Feature Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: feature_distributions.png')

Saved: feature_distributions.png


In [7]:
corr_cols = available_base + ['t_eq', 'stellar_flux']
corr_cols = [c for c in corr_cols if c in df.columns]
corr_data = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(
    corr_data, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax, linewidths=0.5,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: correlation_heatmap.png')

Saved: correlation_heatmap.png


In [8]:
counts_sorted = df['habitable'].value_counts().sort_index()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(
    ['Not Habitable', 'Habitable'], counts_sorted.values,
    color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5, width=0.5
)
axes[0].set_title('Class Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts_sorted.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

scatter_df = df[['pl_orbsmax', 'pl_rade', 'habitable']].dropna()
scatter_df = scatter_df[(scatter_df['pl_orbsmax'] > 0) & (scatter_df['pl_rade'] > 0)]
colors = scatter_df['habitable'].map({0: '#e74c3c', 1: '#2ecc71'})
axes[1].scatter(
    np.log10(scatter_df['pl_orbsmax'].clip(lower=1e-4)),
    scatter_df['pl_rade'], c=colors, alpha=0.35, s=10
)
axes[1].set_xlabel('log10(Semi-major Axis [AU])')
axes[1].set_ylabel('Planet Radius [Earth Radii]')
axes[1].set_title('Orbit vs Radius', fontsize=12, fontweight='bold')

teq_df = df[['t_eq', 'pl_rade', 'habitable']].replace([np.inf, -np.inf], np.nan).dropna()
teq_df = teq_df[(teq_df['t_eq'] > 50) & (teq_df['t_eq'] < 1500)]
colors2 = teq_df['habitable'].map({0: '#e74c3c', 1: '#2ecc71'})
axes[2].scatter(teq_df['t_eq'], teq_df['pl_rade'], c=colors2, alpha=0.35, s=10)
axes[2].axvspan(175, 340, alpha=0.12, color='#2ecc71', label='Habitable T_eq')
axes[2].set_xlabel('Equilibrium Temperature (K)')
axes[2].set_ylabel('Planet Radius [Earth Radii]')
axes[2].set_title('T_eq vs Radius', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('target_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: target_analysis.png')

Saved: target_analysis.png


In [9]:
X = df[all_features].copy()
y = df['habitable'].copy()

X = X.replace([np.inf, -np.inf], np.nan)
valid_mask = ~X[['t_eq', 'stellar_flux', 'hz_ratio']].isna().any(axis=1)
X, y = X[valid_mask].reset_index(drop=True), y[valid_mask].reset_index(drop=True)

print(f'Final dataset: {X.shape}')
print(f'Class distribution: {dict(y.value_counts().sort_index())}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

model = HistGradientBoostingClassifier(
    max_iter=600,
    max_depth=7,
    learning_rate=0.04,
    min_samples_leaf=15,
    max_leaf_nodes=31,
    l2_regularization=0.3,
    class_weight='balanced',
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    random_state=42
)
model.fit(X_train, y_train)
print(f'Iterations used: {model.n_iter_}')

y_pred = model.predict(X_test)
proba  = model.predict_proba(X_test)
model_classes = list(model.classes_)
pos_idx = model_classes.index(1) if 1 in model_classes else None
y_prob  = proba[:, pos_idx] if pos_idx is not None else np.zeros(len(y_test))

acc      = accuracy_score(y_test, y_pred)
bal_acc  = balanced_accuracy_score(y_test, y_pred)

if len(y_test.unique()) > 1 and pos_idx is not None:
    roc_auc      = roc_auc_score(y_test, y_prob)
    avg_precision= average_precision_score(y_test, y_prob)
else:
    roc_auc, avg_precision = float('nan'), float('nan')

print(f'Accuracy:          {acc:.4f}')
print(f'Balanced Accuracy: {bal_acc:.4f}')
print(f'ROC-AUC:           {roc_auc:.4f}')
print(f'Avg Precision:     {avg_precision:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Not Habitable', 'Habitable'], zero_division=0))

Final dataset: (16693, 12)
Class distribution: {0: np.int64(16484), 1: np.int64(209)}
Train: (13354, 12)  |  Test: (3339, 12)
Iterations used: 100
Accuracy:          0.9985
Balanced Accuracy: 0.9757
ROC-AUC:           0.9966
Avg Precision:     0.8984

               precision    recall  f1-score   support

Not Habitable       1.00      1.00      1.00      3297
    Habitable       0.93      0.95      0.94        42

     accuracy                           1.00      3339
    macro avg       0.96      0.98      0.97      3339
 weighted avg       1.00      1.00      1.00      3339



In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_bal = cross_val_score(model, X, y, cv=cv, scoring='balanced_accuracy', n_jobs=-1)
cv_roc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'5-Fold CV Balanced Accuracy: {cv_bal.mean():.4f} +/- {cv_bal.std():.4f}')
print(f'5-Fold CV ROC-AUC:           {cv_roc.mean():.4f} +/- {cv_roc.std():.4f}')

5-Fold CV Balanced Accuracy: 0.9877 +/- 0.0183
5-Fold CV ROC-AUC:           0.9937 +/- 0.0124


In [11]:
# HistGradientBoostingClassifier does not expose feature_importances_;
# use permutation importance on the held-out test set instead.
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model, X_test, y_test,
    n_repeats=30, random_state=42, n_jobs=-1,
    scoring='balanced_accuracy'
)
importances     = perm.importances_mean
importances_std = perm.importances_std

# Clip tiny near-zero values to exactly 0 (within noise floor)
NEGLIGIBLE = 1e-3
importances_clipped = np.where(np.abs(importances) < NEGLIGIBLE, 0.0, importances)

feat_imp_df = pd.DataFrame({
    'feature':    all_features,
    'importance': importances_clipped,
    'std':        importances_std,
    'raw':        importances,
}).sort_values('importance', ascending=True)

FEATURE_LABELS_PLOT = {
    'pl_orbper':   'Orbital Period',
    'pl_rade':     'Planet Radius',
    'pl_masse':    'Planet Mass',
    'pl_orbsmax':  'Semi-major Axis',
    'pl_orbeccen': 'Orbital Eccentricity',
    'st_teff':     'Star Temperature',
    'st_rad':      'Star Radius',
    'st_mass':     'Star Mass',
    'sy_dist':     'Distance (pc)',
    't_eq':        'Equil. Temp [derived]',
    'stellar_flux':'Stellar Flux [derived]',
    'hz_ratio':    'HZ Ratio [derived]',
}
feat_imp_df['label'] = feat_imp_df['feature'].map(lambda f: FEATURE_LABELS_PLOT.get(f, f))

significant = feat_imp_df[feat_imp_df['importance'] >= NEGLIGIBLE]
negligible  = feat_imp_df[feat_imp_df['importance'] <  NEGLIGIBLE]

has_negligible = len(negligible) > 0
n_panels = 2 if has_negligible else 1
width_ratios = [3, 1] if has_negligible else [1]

fig, axes_list = plt.subplots(
    1, n_panels, figsize=(14 if has_negligible else 9, 7),
    gridspec_kw={'width_ratios': width_ratios}
)
if n_panels == 1:
    axes_list = [axes_list]

# --- Left panel: significant features ---
ax = axes_list[0]
sig_colors = plt.cm.plasma(np.linspace(0.25, 0.85, max(len(significant), 1)))
bars = ax.barh(
    significant['label'], significant['importance'],
    xerr=significant['std'], color=sig_colors, height=0.55,
    error_kw=dict(ecolor='#444', capsize=4, linewidth=1.2)
)
x_max = significant['importance'].max() if len(significant) else 0.1
for bar, val, std in zip(bars, significant['importance'], significant['std']):
    ax.text(val + std + x_max * 0.02,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, x_max * 1.25)
ax.set_xlabel('Permutation Importance\n(Balanced Accuracy drop when shuffled)', fontsize=10)
ax.set_title('Significant Features', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Right panel: negligible features (dot plot) ---
if has_negligible:
    ax2 = axes_list[1]
    y_pos = list(range(len(negligible)))
    ax2.scatter(negligible['raw'], y_pos, color='#7f8c8d', s=55, zorder=3)
    ax2.errorbar(
        negligible['raw'], y_pos,
        xerr=negligible['std'],
        fmt='none', ecolor='#bdc3c7', capsize=3, linewidth=1
    )
    ax2.axvline(0, color='#e74c3c', linewidth=1.2, linestyle='--', alpha=0.7, label='zero')
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(negligible['label'], fontsize=9)
    ax2.set_xlabel('Raw importance value', fontsize=9)
    ax2.set_title('Negligible Features\n(|importance| < 0.001)', fontsize=11, fontweight='bold')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

plt.suptitle('Feature Importance — HistGradientBoosting (Permutation)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')
print('\nAll features (sorted by importance):')
print(feat_imp_df[['feature','importance','std']].sort_values('importance', ascending=False).to_string(index=False))


Saved: feature_importance.png

All features (sorted by importance):
     feature  importance      std
        t_eq    0.433455 0.019389
     pl_rade    0.271024 0.038664
     st_teff    0.050273 0.022508
   pl_orbper    0.000000 0.000000
 pl_orbeccen    0.000000 0.000000
  pl_orbsmax    0.000000 0.000000
    pl_masse    0.000000 0.000000
     st_mass    0.000000 0.000000
     sy_dist    0.000000 0.000000
stellar_flux    0.000000 0.000000
      st_rad    0.000000 0.000000
    hz_ratio    0.000000 0.000000


In [12]:
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['Not Habitable', 'Habitable'],
    yticklabels=['Not Habitable', 'Habitable'],
    linewidths=0.5
)
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

if len(y_test.unique()) > 1 and pos_idx is not None:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[1].plot(fpr, tpr, color='#2980b9', linewidth=2.5, label=f'ROC-AUC = {roc_auc:.3f}')
    axes[1].fill_between(fpr, tpr, alpha=0.1, color='#2980b9')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
    axes[1].legend(loc='lower right', fontsize=10)
    fpr_list = fpr.tolist()
    tpr_list = tpr.tolist()

    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    axes[2].plot(recall, precision, color='#8e44ad', linewidth=2.5,
                 label=f'Avg Precision = {avg_precision:.3f}')
    axes[2].fill_between(recall, precision, alpha=0.1, color='#8e44ad')
    axes[2].legend(loc='upper right', fontsize=10)
else:
    for ax in axes[1:]:
        ax.text(0.5, 0.5, 'Not available', ha='center', va='center', transform=ax.transAxes)
    fpr_list, tpr_list = [], []

axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('model_performance.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: model_performance.png')

Saved: model_performance.png


In [13]:
feature_stats = {}
for col in all_features:
    col_data = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    feature_stats[col] = {
        'mean':   float(col_data.mean()),
        'median': float(col_data.median()),
        'std':    float(col_data.std()),
        'min':    float(col_data.min()),
        'max':    float(col_data.max()),
        'q25':    float(col_data.quantile(0.25)),
        'q75':    float(col_data.quantile(0.75))
    }

model_bundle = {
    'model':              model,
    'model_name':         'HistGradientBoostingClassifier',
    'base_feature_cols':  available_base,
    'derived_feature_cols': DERIVED_FEATURES,
    'all_feature_cols':   all_features,
    'feature_stats':      feature_stats,
    'accuracy':           acc,
    'balanced_accuracy':  bal_acc,
    'roc_auc':            roc_auc,
    'avg_precision':      avg_precision,
    'cv_bal_acc_mean':    float(cv_bal.mean()),
    'cv_bal_acc_std':     float(cv_bal.std()),
    'cv_roc_mean':        float(cv_roc.mean()),
    'cv_roc_std':         float(cv_roc.std()),
    'report': classification_report(
        y_test, y_pred,
        target_names=['Not Habitable', 'Habitable'],
        output_dict=True, zero_division=0
    ),
    'confusion_matrix':   cm.tolist(),
    'roc_curve':          {'fpr': fpr_list, 'tpr': tpr_list},
    'feature_importance': feat_imp_df.set_index('feature')['importance'].to_dict(),
    'target_distribution': df['habitable'].value_counts().to_dict(),
}

with open('model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)

size_mb = os.path.getsize('model.pkl') / 1024 / 1024
print(f'model.pkl saved  |  Size: {size_mb:.2f} MB')
print(f'Accuracy: {acc:.4f}  |  Balanced Acc: {bal_acc:.4f}  |  ROC-AUC: {roc_auc:.4f}')
print(f'Base features:    {available_base}')
print(f'Derived features: {DERIVED_FEATURES}')

model.pkl saved  |  Size: 0.09 MB
Accuracy: 0.9985  |  Balanced Acc: 0.9757  |  ROC-AUC: 0.9966
Base features:    ['pl_orbper', 'pl_rade', 'pl_masse', 'pl_orbsmax', 'pl_orbeccen', 'st_teff', 'st_rad', 'st_mass', 'sy_dist']
Derived features: ['t_eq', 'stellar_flux', 'hz_ratio']
